# A worked sweep: the flip agent on the flip demo set

Ten SLURM jobs of five arms each — 50 configurations over 400 plays.

| knob | batch | drawn from | observations this sweep gets |
|---|---|---|---|
| `answers` | **arm** (one per pass) | five answer sequences | **50** — five per job, a fresh block per job |
| `max_concurrent` | **run** (one per job) | 2 or 8 | **10** — one per job, however many games run inside it |

That asymmetry is the whole reason for two batches. Arms are nearly free: they ride
along inside a job that was going to run anyway. A run-level knob costs a job per
observation, which is why hosting and engine settings are swept across pooled jobs
rather than inside one.

CPU-only — no LLM, no GPU. Every cell is re-runnable: submitted jobs are not
resubmitted, finished ones are reused.

In [ ]:
import asyncio
import getpass
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from sweeper import DEFAULT_BUDGET_TOKENS, by_arm, by_job, collect_rows, unit_for
from sweeper.example_analysis import (
    DESIGN,
    arm_seeds_for,
    deploy_job,
    finished_job_dirs,
    job_status,
)

# Shared storage: a SLURM worker writes its artifacts from a compute node.
RUNS_ROOT = Path("/data/shared/research/arc_2_results") / getpass.getuser() / "sweeper_example"
RUN_SEEDS = tuple(range(10))

# Every arm is scored at ONE spend, not where it happened to stop. The flip
# agent spends no tokens, so this changes nothing here — it is the same call an
# LLM sweep depends on.
BUDGET_TOKENS = DEFAULT_BUDGET_TOKENS
# After the imports, not before: taaf.diagnostics calls
# matplotlib.use('Agg') at import time for headless rendering, so a
# backend chosen earlier is silently replaced and no figure ever shows.
%matplotlib inline

## Look at the design before spending anything

The whole design is a function of the seeds, so it is knowable before a single core
is allocated. Worth checking that the arms cover the choices — 50 draws over
five sequences can still clump — and that the run-level knob is not degenerate, which
would spend ten jobs on one value.

This inspects the *inputs* only. Choosing seeds by looking at outcomes would be
picking the answer.

In [ ]:
design = pd.DataFrame(
    [
        {
            "run_seed": run_seed,
            "arm_seed": arm_seed,
            **DESIGN.run_values_for(run_seed),
            **DESIGN.arm_values_for(arm_seed),
        }
        for run_seed in RUN_SEEDS
        for arm_seed in arm_seeds_for(run_seed)
    ]
)

print(f"{len(design)} arms across {design['run_seed'].nunique()} jobs")
print("arms per answer sequence:", dict(design["answers"].astype(str).value_counts()))
print("jobs per max_concurrent:", dict(design.groupby("max_concurrent")["run_seed"].nunique()))
design.head()

## Submit

One SLURM job per run seed, fire and forget. Already-submitted and already-finished
jobs are left alone, so this cell is safe to re-run — which is also how an
interrupted sweep is resumed.

In [ ]:
# Paced, not fired at once. A worker's first `uv sync` on a node builds the
# shared Python install, and jobs that land together race it: the loser
# starts against a half-built interpreter and dies before pytest. TAAF's
# deploy does not heal that (`roadmap.md`), and the pattern this example
# teaches is the one production uses, where the cost is real.
for run_seed in RUN_SEEDS:
    await deploy_job(RUNS_ROOT, run_seed)
    await asyncio.sleep(10)

## Watch

Status is read off the job dirs rather than off handles, so this survives a kernel
restart and works from any machine that can see the shared filesystem. Interrupt it
whenever — the jobs are on the cluster, not in this process.

In [ ]:
while True:
    states = pd.Series([job_status(RUNS_ROOT, seed) for seed in RUN_SEEDS]).value_counts()
    print(f"{time.strftime('%H:%M:%S')} {dict(states)}", flush=True)
    if states.get("finished", 0) == len(RUN_SEEDS):
        break
    time.sleep(60)

## Collect

One row per (arm, game). Only finished jobs are read, so this runs on a partial
sweep too — it simply has fewer observations.

In [ ]:
job_dirs = finished_job_dirs(RUNS_ROOT, RUN_SEEDS)
rows = collect_rows(job_dirs, budget_tokens=BUDGET_TOKENS)
df = pd.DataFrame([row.to_dict() for row in rows])
print(
    f"{len(df)} rows from {len(job_dirs)}/{len(RUN_SEEDS)} finished jobs: "
    f"{df['game_id'].nunique()} games x {df['arm_seed'].nunique()} arms"
)
df.head()

## Aggregate to the unit each knob was drawn at

Every knob is analysed where its draw happened, and nowhere else. `answers` is
drawn per arm, so an arm is one observation of it; `max_concurrent` is drawn per
job, so a job is one observation of it however many arms and games sat inside.

Getting this wrong is the easiest way to manufacture significance: 400 games
carrying ten distinct draws of a run-level knob would report `n` in the hundreds.

In [ ]:
ARM_KNOBS = sorted(DESIGN.arm)
RUN_KNOBS = sorted(DESIGN.run)
KNOBS = ARM_KNOBS + RUN_KNOBS
OUTCOMES = ["score", "actions"]

# The aggregation itself lives in the package: which unit a knob's
# observations are counted at follows from the design, not from taste, and
# getting it wrong is what turns ten decisions into a three-digit n.
by_arm_df = pd.DataFrame([row.to_dict() for row in by_arm(rows)])
by_job_df = pd.DataFrame([row.to_dict() for row in by_job(rows, DESIGN)])
# A sequence-valued knob groups by its label; a tuple is not a category pandas
# will box-plot by.
by_arm_df["answers"] = by_arm_df["answers"].map(lambda answers: "+".join(answers))
UNITS = {knob: (by_arm_df if unit_for(DESIGN, knob) == "arm" else by_job_df, unit_for(DESIGN, knob)) for knob in KNOBS}

print(f"{len(by_arm_df)} arms, {len(by_job_df)} jobs")
by_arm_df.head()

## Every knob against every outcome

The panel per (knob, outcome) a sweep is read from: box plots where a knob has few
values, a scatter on a log axis where it spans decades, and the regression p-value
in the title (an ANOVA over the levels for a knob whose values are labels). `n` in each title counts observations *of that knob*, which is why
the run-level column is so much thinner than the per-arm one.

In [ ]:
def knob_pvalue(frame, knob, outcome):
    """Regression p-value of the outcome on a numeric knob (for a binary knob
    the two-sample t-test); a one-way ANOVA over the levels of a labelled one."""
    y = frame[outcome].astype(float)
    values = frame[knob]
    if pd.api.types.is_numeric_dtype(values):
        x = pd.to_numeric(values, errors="coerce").astype(float)
        ok = x.notna() & y.notna()
        if ok.sum() < 3 or x[ok].nunique() < 2:
            return float("nan")
        return float(stats.linregress(x[ok], y[ok]).pvalue)
    groups = [y[(values == level) & y.notna()] for level in values.unique()]
    groups = [g for g in groups if len(g) > 1 and g.nunique() > 1]
    if len(groups) < 2:
        return float("nan")
    return float(stats.f_oneway(*groups).pvalue)


def plot_knob(ax, frame, knob, outcome, unit):
    """One panel: a knob's values against an outcome, at the knob's own unit."""
    values = frame[knob]
    levels = sorted(values.unique())
    if len(levels) <= 5:
        ax.boxplot([frame.loc[values == level, outcome] for level in levels], tick_labels=[str(v) for v in levels])
    else:
        ax.scatter(values, frame[outcome], alpha=0.7)
        if pd.api.types.is_numeric_dtype(values) and values.min() > 0 and values.max() / values.min() > 20:
            ax.set_xscale("log")
    p = knob_pvalue(frame, knob, outcome)
    stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    ax.set(xlabel=f"{knob} ({unit})", ylabel=outcome)
    ax.set_title(f"{outcome} vs {knob} - p={p:.3f}{stars}, n={len(frame)} {unit}s", fontsize=10)
    ax.grid(alpha=0.3)


fig, axes = plt.subplots(len(OUTCOMES), len(KNOBS), figsize=(5.5 * len(KNOBS), 4 * len(OUTCOMES)), squeeze=False)
for row, outcome in enumerate(OUTCOMES):
    for col, knob in enumerate(KNOBS):
        frame, unit = UNITS[knob]
        plot_knob(axes[row][col], frame, knob, outcome, unit)
fig.tight_layout()

## The per-arm knob in detail

One box per answer sequence, with the error bars that say whether the arm-to-arm
spread is bigger than the noise inside an arm (SEM over that arm's 8 games).
Without them a scatter of arm means invites reading structure into 8-game noise.

The second panel is the mechanism check: every arm submitted exactly as many
answers as its sequence holds — which is what shows the per-arm value reached the
game rather than only the record.


In [ ]:
knob = ARM_KNOBS[0]
lengths = by_arm_df[knob].map(lambda label: label.count("+") + 1)
assert (by_arm_df["actions"] == lengths).all(), "an arm submitted a different number of answers than its sequence"
print(f"{by_arm_df[knob].nunique()} distinct sequences over {len(by_arm_df)} arms")

fig, (ax_fit, ax_cost) = plt.subplots(1, 2, figsize=(12, 4.5))
levels = sorted(by_arm_df[knob].unique())
for value, group in by_arm_df.groupby(RUN_KNOBS[0]):
    ax_fit.errorbar(
        group[knob].map(levels.index),
        group["score"],
        yerr=group["score_sem"],
        fmt="o",
        alpha=0.75,
        capsize=3,
        label=f"{RUN_KNOBS[0]}={value}",
    )
    ax_cost.scatter(group["actions"], group["score"], alpha=0.75, label=f"{RUN_KNOBS[0]}={value}")
ax_fit.set_xticks(range(len(levels)), levels, rotation=20, ha="right")
ax_fit.set(xlabel=knob, ylabel="mean score (+/- SEM)", title="Score vs the arm's answer sequence")
ax_cost.set(xlabel="mean answers submitted", ylabel="mean score", title="Score vs what it cost")
for ax in (ax_fit, ax_cost):
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()

## Distributions and correlations

The spread of arm means says what a single arm is worth as evidence. The
correlation matrix carries the labelled knob as the number of answers in its
sequence, the one numeric thing about it.

In [ ]:
matrix = pd.DataFrame(
    {
        f"len({ARM_KNOBS[0]})": by_arm_df[ARM_KNOBS[0]].map(lambda label: label.count("+") + 1),
        **{run_knob: by_arm_df[run_knob].astype(float) for run_knob in RUN_KNOBS},
        "score": by_arm_df["score"],
        "actions": by_arm_df["actions"],
    }
).corr()

fig, (ax_hist, ax_heat) = plt.subplots(1, 2, figsize=(12, 4.5))
ax_hist.hist(by_arm_df["score"], bins=20, edgecolor="black", alpha=0.75)
ax_hist.axvline(by_arm_df["score"].mean(), color="red", ls="--", label=f"mean {by_arm_df['score'].mean():.3f}")
ax_hist.axvline(by_arm_df["score"].median(), color="green", ls="--", label=f"median {by_arm_df['score'].median():.3f}")
ax_hist.set(xlabel="arm mean score", ylabel="arms", title="How much arms differ")
ax_hist.legend()
ax_hist.grid(alpha=0.3)

image = ax_heat.imshow(matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax_heat.set_xticks(range(len(matrix)), matrix.columns, rotation=30, ha="right")
ax_heat.set_yticks(range(len(matrix)), matrix.columns)
for i in range(len(matrix)):
    for j in range(len(matrix)):
        ax_heat.text(j, i, f"{matrix.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
ax_heat.set_title("Correlations at arm level")
fig.colorbar(image, ax=ax_heat, shrink=0.8)
fig.tight_layout()

## Best and worst arms

The table a sweep is usually read for, and the one most likely to be
over-interpreted. With 8 games an arm, the gap between the best arm and the median
sits comfortably inside the error bars above: what generalises is the trend across
levels, not the winner.

In [ ]:
columns = ["run_seed", "arm_seed", *KNOBS, "score", "score_sem", "actions"]
print("BEST 5 ARMS")
display(by_arm_df.nlargest(5, "score")[columns].reset_index(drop=True))
print("WORST 5 ARMS")
display(by_arm_df.nsmallest(5, "score")[columns].reset_index(drop=True))

## The run-level knob: ten observations, one per job

`max_concurrent` is drawn once per job, so **the job is the unit** however many games
ran inside it. Testing across arms or across games would count one draw dozens of
times and manufacture a p-value out of nothing.

In [ ]:
levels = sorted(by_job_df[RUN_KNOBS[0]].unique())
groups = [by_job_df.loc[by_job_df[RUN_KNOBS[0]] == value, "score"] for value in levels]
if all(len(group) >= 2 for group in groups):
    test = stats.ttest_ind(groups[1], groups[0], equal_var=False)
    print(
        f"{RUN_KNOBS[0]}={levels[-1]} vs {levels[0]}: {groups[-1].mean() - groups[0].mean():+.3f} "
        f"(p={test.pvalue:.3f}, n={len(groups[1])} vs {len(groups[0])} JOBS)"
    )
else:
    print("one value has fewer than two jobs - add run seeds")

jitter = np.random.default_rng(0).uniform(-0.06, 0.06, len(by_job_df))
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.boxplot(groups, tick_labels=[str(level) for level in levels])
offset = 0
for index, group in enumerate(groups, start=1):
    ax.scatter(np.full(len(group), index) + jitter[offset : offset + len(group)], group, alpha=0.8, zorder=3)
    offset += len(group)
ax.set(xlabel=RUN_KNOBS[0], ylabel="job mean score", title="One point per job - all a run-level knob gets")
ax.grid(alpha=0.3)
fig.tight_layout()

## What this does not show

- The agent is scripted, so `answers` decides the score outright and
  `max_concurrent` cannot touch it. The point is the machinery, not the finding.
- Fifty arms against one knob is a comfortable design; against a dozen knobs it is
  thin again, and the answer is more jobs rather than more arms.
- TAAF's own `diagnostics.html` for these jobs averages across arms and reports a σ
  across passes that is really the spread between configurations. It is not a result
  for a sweep run — this notebook is.